# Exotica `.0000.h5` — distributional deep-dive & training-set decisions

Follow-up to `04_exotica_0000_real_data.ipynb` (batch inventory, cadence structure, band coverage — purely descriptive, explicitly deferred judging `configs/data/gbt_fine.yaml`). This notebook **is** that judgment step: it answers the three questions that actually gate building the training dataset:

1. **Normalization** — does `bandpass_correct(poly_degree=3)` + `core_transform` (log1p + median/MAD) survive contact with real noise/RFI, or is a change (e.g. arcsinh) needed? (open question tracked in memory: unbounded median/MAD, RFI ~100× vs noise [-4,8] was the worst-case worry pre-real-data)
2. **RFI characterization** — occupancy fraction and intensity per band, since RFI (not signal) is the known failure mode that dominates reconstruction error (see `docs/decisions/scoring-history.md` §1).
3. **Band/cadence completeness → usable training budget**, and **ON vs OFF pixel-distribution comparison** → is it safe to train the unsupervised scorer on the full batch (Exotica sources are both the training source and the Phase-3 search target), or must ON sources be held out.

All numbers below come from `scripts/debug/exotica_dist_analysis.py`, run on the data host (`docker exec neurabeast_filippo`, mount `/content/wd_mybook`) against the real batch — not synthetic data. It samples 8 representative bands (1 complete 6-file cadence each, both an ON and an OFF file), 3 sub-bands per file (4096 chan windows at 10%/50%/90% of each file's covered range), rather than loading all ~5400 files.

## 0. Header assumptions — confirmed against real headers

Verify shapes/df/dt against real `.h5` headers, not spec docs.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"pyproject.toml not found above {start}")


REPO_ROOT = find_repo_root(Path.cwd())
DIST_DIR = REPO_ROOT / "data" / "processed" / "dist_analysis"
summary = json.loads((DIST_DIR / "summary.json").read_text())
stats_df = pd.read_csv(DIST_DIR / "subband_stats.csv")
by_band = pd.read_csv(DIST_DIR / "complete_cadences_per_band.csv")

print(json.dumps(summary, indent=2))

**Result**: across all 5432 valid files, `nchans` = 67,108,864, `ntime` = 16, `df` = -2.794 Hz/chan, `dt` = 18.254 s/bin, `duration` = 292.06 s — every single value, zero spread. This exactly matches `configs/data/gbt_fine.yaml`'s `raw.nchans`/`raw.df`/`raw.dt` (2.7939677238464355 Hz, 18.25361108 s). Only `duration_s` (292.06 s) differs slightly from the spec's quoted ~308 s — real headers win over spec docs; `tchans=96` (6×16) is unaffected since it's bin-count driven, not duration driven.

## 1 & 2. Normalization + RFI — does the current pipeline survive real data?

In [ ]:
cols = ["band","role","target","subband_pos","raw_median","rfi_frac","rfi_intensity_ratio",
        "post_transform_min","post_transform_max","post_transform_p01","post_transform_p99"]
stats_df[cols].sort_values("post_transform_max", ascending=False).head(10)

**Normalization result on this sample — encouraging, but not a full resolution of the open question.** A known open question documents specific training frames where a strong narrowband RFI line pushes `post_transform` to ~100 against a noise floor of ~[-4, 8], while noting that finding stands on a fixed RFI-line case, not a survey. This notebook's 8-band / 48-subband real-data sample never hits that regime:

- `post_transform` ranges **[-6.6, +19.4]**, median sub-band max around **+6.6**, only one outlier sub-band reaching +19.4.
- p01/p99 sit at **[-3.4, +3.5]** consistently — a clean, symmetric, near-Gaussian bulk, matching the noise-only expectation.
- RFI occupancy in this sample is tiny (<0.4% of channels even at the high end, see below) — too sparse in these particular sub-bands to reproduce the ~100 amplitude case from memory, which likely requires either a denser/brighter RFI band or a different frequency/window than the 8 sampled here.

**Conclusion: no evidence here to change `poly_degree=3` + log1p + median/MAD before building the training set** — consistent with the existing verdict that the normalization issue is real-but-second-order, not a blocker. This does *not* retire the open question (memory's arcsinh candidate remains queued for a controlled ablation); it just means a random band survey doesn't reproduce the worst case, so the fix stays a hedge rather than an urgent hotfix.

In [ ]:
display(Image(str(DIST_DIR / "distributions_summary.png")))

**RFI occupancy** (channels flagged >5σ-robust above the per-subband median-of-medians baseline, within 4096-chan / ~11.4 kHz windows):

- Mean flagged fraction: **0.022% (ON)** vs **0.011% (OFF)** — RFI is sparse at this frequency resolution/window size in both roles; the right-hand histogram above shows almost all sub-bands at ~0 flagged channels, with a few outliers up to 0.37% (ON) / 0.12% (OFF).
- ON files show roughly double the RFI rate of OFF, plausible if Exotica targets sit in more RFI-congested parts of the band or simply from small-sample variance (n=24 sub-bands per role) — not large enough to demand per-role RFI handling.
- No evidence of the "static RFI spikes error, smooth signal lowers it" failure mode (see `docs/decisions/scoring-history.md` §1) needing a fix at the normalization stage — the RFI here is too sparse to noticeably skew the bulk distribution.

## 3. Band / cadence completeness — training budget

In [ ]:
print(f"{summary['n_complete_session_bands']}/{summary['n_session_band_entries']} "
      "session-band entries are complete 6-file cadences")
by_band.columns = ["band", "n_complete_cadences"]
by_band.sort_values("n_complete_cadences", ascending=False).head(20)

**Result (CORRECTED 2026-07-06 — see section 4 below for the full story)**: an earlier bucketing bug undercounted cadences by grouping only on (session, rounded band), which silently merged multiple distinct targets observed back-to-back on the same node/band, and conflated genuinely different node-pairs (e.g. `blc17`/`blc21`) that cover identical nominal bands with different real data. After fixing the grouping to include node identity and chunk consecutive 6-file runs, the batch has **905 complete cadences** (not 358) across **22 distinct ON targets** (not 14), spanning the same 750 MHz-12.4 GHz range but with much better per-band coverage (minimum 7 cadences/band, not 3). See section 4 for the corrected breakdown and the root-cause explanation.

## ON vs OFF — is training on the full batch (including Exotica sources) safe?

Ran a KS test on raw pixel values, ON (Exotica target, repeated 3× per cadence) vs OFF (reference star, singleton), pooled across the 8 sampled bands (~200k-pixel subsample each side):

```
raw pixel KS test ON vs OFF: stat=0.0191  p=5.69e-32
ON median=2.638e6   OFF median=2.694e6
```

The difference is statistically significant (large n makes p tiny almost by default) but the **effect size is negligible** (KS D=0.019; medians within 2% of each other) and the post-`core_transform` histograms (plot above) are visually indistinguishable between ON and OFF. **Conclusion: training the unsupervised scorer on the full batch — both ON (Exotica sources) and OFF (reference stars) — is safe.** Any true technosignature is rare enough not to shift these bulk statistics, so contaminating the "normal" manifold with occasional ON-source anomalies is not a practical concern at this sample size. No need to hold out ON files from training.

## Summary — decisions for building the training dataset

| Question | Answer |
|---|---|
| Bandpass/normalization (`poly_degree=3`, log1p+median/MAD) | **Keep as-is for now.** This 8-band survey saw RFI blow-up of ~3-19× MAD, not the ~100× worst case from memory — but that worst case came from a specific strong-RFI frame, not a survey, so the open question (arcsinh candidate) stays queued rather than closed. |
| RFI occupancy | Sparse (<0.4% of channels per sub-band even at the high end); no special RFI-aware normalization needed at this product's resolution. |
| Training data source | Real GBT Exotica batch replaces SRT/synthetic as primary training source for `.0000.fil` — real noise statistics match the pipeline's design assumptions closely enough that there's no blocker. |
| Which cadences to include | 358 complete 6-file cadences across 49 bands — build `data/raw/gbt_0000_cadences.txt` from all complete session-bands (script above already enumerates them), stratifying train/val split by band so no receiver setup is val-only. |
| ON/OFF handling | Train on everything (ON + OFF); ON-vs-OFF pixel distributions are statistically distinguishable but practically identical (KS D=0.019). |

**Next step**: extend `scripts/debug/exotica_dist_analysis.py`'s session-band enumeration into a cadence-list builder (mirroring `data/raw/srt_0000_cadences.txt`'s format) to actually populate `dataset.cadence_list` in `configs/data/gbt_fine.yaml`, then run the existing `build_datasets()` cache pipeline against it.

## 4. Cadence census: completeness, target repetition, frequency distribution (corrected)

Follow-up questions (2026-07-06): how many complete cadences exist, whether the same target repeats across cadences/time/frequency, and how cadences distribute over frequency. First pass grouped files by `(session, rounded band)` only and found 358 complete cadences / 240 "incomplete" (`n_files != 6`) — but the "incomplete" group turned out to be `n_files ∈ {12, 18, 24}`, multiples of 6, not fragments.

**Root cause, found by inspecting the data directly**: two things were being conflated under one bucket key:
1. **Node-pair collisions**: fixed node pairs (e.g. `blc17`/`blc21`) cover the *exact same* nominal band at the *exact same* timestamp. Content diff confirmed these are genuinely different data (different file size, ~60x different amplitude scale, uncorrelated values) — not duplicates, likely two backend/polarization chains — so both must count as independent cadences, not be merged or deduplicated.
2. **Multi-target sequences**: a single node/band can carry more than one Exotica target back-to-back within one observing session (e.g. `MESSIER32` ON/OFF/ON/OFF/ON/OFF immediately followed by `NGC891` ON/OFF/ON/OFF/ON/OFF on the same node/band) — the original grouping inferred one dominant ON target across the whole inflated group instead of recognizing two separate cadences.

**Fix**: group by `(session, rounded band, node)` and split each group into consecutive 6-file chunks. Result: **905 complete cadences** (2.5x more than the buggy count), only 2 leftover stray files in the whole batch, and every chunk resolves to a clean `ON×3/OFF×3 (3 different OFF sources)` pattern — no ambiguity. Computed by `scripts/debug/exotica_cadence_summary.py`.

In [ ]:
import subprocess
print(subprocess.run(["python3", str(REPO_ROOT / "scripts/debug/exotica_cadence_summary.py")],
                      capture_output=True, text=True, cwd=REPO_ROOT).stdout)

**How many complete cadences.** 905, across 48 real observing sessions and node/band-specific chunks. Minimum per-band coverage rose from 3 to **7** — the thin-band problem that motivated stratified splitting is real but much less severe than first thought.

**Target repetition.** **22 distinct ON (Exotica) targets** (not 14) — the multi-target-per-node/band fix surfaced targets that were previously hidden inside merged buckets (e.g. `3C273`, `3C279`, `MESSIER100`, `IC3877`, `NGC7649`, `NGC1068`, `WD0135-052`, `HD220077`). All 22 now show >1 distinct MJD day (3-5 each) — every target has genuine multi-epoch repetition, not just the parallel-band artifact.

**Cadence distribution vs frequency**: still uneven but far less extreme — L-band now has 8-22 cadences/band (was 3-8), concentration remains in C-high/X-low (5-8GHz: 300 cadences/16 bands) and X-mid (8-10GHz: 127/11 bands).

In [ ]:
display(Image(str(DIST_DIR / "cadences_by_band.png")))

**Implication for the earlier stratified-split plan**: the band/target confound is somewhat softer now (each target spans many more bands and days), but the core plan from the /grill-me session stands — stratify by both band and target (explicit minimum-coverage constraint in both, agreed for target given only 22 distinct values), verify day-diversity post-hoc rather than constraining it upfront. The corrected `all_complete_cadences.csv` (905 rows, with full file lists) is the input to the next step: the stratified manifest builder (`scripts/build_gbt_cadence_manifest.py`, not yet written) that will produce `train`/`val`/`held-out` cadence lists for `configs/data/gbt_fine.yaml`.

## 5. Exploratory plots for split planning (pre-implementation, `/grill-me` session 2026-07-06)

Before writing the stratified manifest builder, four plots from `all_complete_cadences.csv` (905 rows) to see the actual shape of the band/target/time coverage the split has to respect.

In [ ]:
display(Image(str(DIST_DIR / "target_x_band_heatmap.png")))

**Target x band coverage matrix.** Only **7 of 22 targets** (`OJ287`, `MESSIER67`, `HIP114176`, `3C125`, `GJ299`, `NGC4151`, `NGC891` — also the ones with the most total cadences) have any X-band coverage (8-12.4 GHz); the other 15 stop around 7 GHz. This is a real absence in the batch, not a split artifact — no amount of stratification can put X-band held-out cadences for those 15 targets, because none exist. Any claim about "searching the full spectrum for target X" needs this caveat for 15/22 targets.

In [ ]:
display(Image(str(DIST_DIR / "cadences_per_target.png")))

**Cadences per target**, sorted: 19 (the 8 thinnest targets) to 76 (`OJ287`, `MESSIER67`). Comfortably above the >=1-per-pool constraint for every target.

In [ ]:
display(Image(str(DIST_DIR / "days_per_band.png")))

**Temporal diversity per band.** Most bands span 4-12 distinct MJD days — but two bands (750-939 MHz, 2064-2251 MHz) have only **1** distinct day. These two bands will unavoidably fail the post-hoc day-diversity check discussed earlier (every pool that includes them gets the same single day) — not fixable by resplitting, just a known caveat to note when interpreting results from those bands.

In [ ]:
display(Image(str(DIST_DIR / "cadences_per_session_by_region.png")))

**Cadences per session by frequency region.** Confirms the receiver-setup blocking already seen in notebook 04: X-band sessions cluster at the start/end of the observing timeline, L-band in an early-middle block, S-band and C-high/X-low in their own blocks — consistent with GBT receivers needing physical swaps between bands, not observed continuously.